# Intrinsic evaluation and random sentence generation (Task 2 and Task 1.8)

Amharic n-gram language model, NLP Assignment 1.

This notebook (1) downloads the processed data and the team's code, (2) trains the full n-gram model, (3) computes perplexity for n = 1 to 4 on the test set with several estimation methods, (4) generates random sentences from each model, and (5) checks whether the generated sentences are copied from the training data. The evaluation code is written to `lm_eval.py`.

## 1. Setup: data, code, full model

In [1]:
!pip install -q gdown
import gdown
gdown.download_folder(
    "https://drive.google.com/drive/folders/1gnliA5u4iXgDZ2cKBO776K0NXi5PlL33",
    output="/kaggle/working/processed", quiet=False)


Retrieving folder contents


Processing file 1DhcnLXkHDrZRJB4E4ok4OeLvmHvyBFSX .DS_Store
Processing file 1hFDW9wTwxom9Q-LaKbKnupWSKfcToZHL dev.npy
Processing file 1bZ8GyCMQYQowIta4R1mjWl0RK2tXCJ-U stats.json
Processing file 18HipZ-GJ7o1XL6590f7jdxZm1UloeUOB test.npy
Processing file 18xK-8M_u7Pn0yiTh08wsg6YnVx9XNfXu train.npy
Processing file 1brf_NnFPvMUAxgaMBVK4POiWOKxjgDL0 vocab.txt


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1DhcnLXkHDrZRJB4E4ok4OeLvmHvyBFSX
To: /kaggle/working/processed/.DS_Store
100%|██████████| 6.15k/6.15k [00:00<00:00, 5.43MB/s]
Downloading...
From: https://drive.google.com/uc?id=1hFDW9wTwxom9Q-LaKbKnupWSKfcToZHL
To: /kaggle/working/processed/dev.npy
100%|██████████| 34.4M/34.4M [00:00<00:00, 101MB/s] 
Downloading...
From: https://drive.google.com/uc?id=1bZ8GyCMQYQowIta4R1mjWl0RK2tXCJ-U
To: /kaggle/working/processed/stats.json
100%|██████████| 2.76k/2.76k [00:00<00:00, 3.58MB/s]
Downloading...
From: https://drive.google.com/uc?id=18HipZ-GJ7o1XL6590f7jdxZm1UloeUOB
To: /kaggle/working/processed/test.npy
100%|██████████| 34.3M/34.3M [00:00<00:00, 121MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=18xK-8M_u7Pn0yiTh08wsg6YnVx9XNfXu
From (redirected): https://drive.google.com/uc?id=18xK-8M_u7Pn0yiTh08wsg6YnVx9XNfXu&

['/kaggle/working/processed/.DS_Store',
 '/kaggle/working/processed/dev.npy',
 '/kaggle/working/processed/stats.json',
 '/kaggle/working/processed/test.npy',
 '/kaggle/working/processed/train.npy',
 '/kaggle/working/processed/vocab.txt']

In [3]:
!sha256sum /kaggle/working/processed/train.npy /kaggle/working/processed/vocab.txt

c88588aa527108a3376676c355c212dc79136fcfb879f7a4c54796d070dabb42  /kaggle/working/processed/train.npy
84d8c01602e96fc132bd109d578adaf796b23f1c4aeed696a911b46e7f798e7c  /kaggle/working/processed/vocab.txt


In [2]:
!git clone -b member2-ngram-modeling https://github.com/NaomiMeseret/amharic-ngram-language-model /kaggle/working/code
!ls /kaggle/working/code/src

Cloning into '/kaggle/working/code'...
remote: Enumerating objects: 227, done.
remote: Counting objects: 100% (227/227), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 227 (delta 85), reused 219 (delta 82), pack-reused 0 (from 0)
Receiving objects: 100% (227/227), 420.86 KiB | 3.54 MiB/s, done.
Resolving deltas: 100% (85/85), done.
ngram_disk.py	preprocess_lib.py  train_ngram_slice.py
ngram_model.py	preprocess.py


In [3]:
import sys, numpy as np
sys.path.insert(0, "/kaggle/working/code")
from src.ngram_model import NGramModel
from src.ngram_disk import DiskBackedNGramTrainer
train = np.load("/kaggle/working/processed/train.npy", mmap_mode="r")
print(train.shape, train.dtype, train[:12])

(68942614,) int32 [    2  8947    47   608 18124     1     1     1   217  1391   374  9216]


In [4]:
script = '''
import sys
sys.path.insert(0, "/kaggle/working/code")
from src.ngram_disk import DiskBackedNGramTrainer
t = DiskBackedNGramTrainer(
    "/kaggle/working/processed/train.npy", "/kaggle/working/processed/vocab.txt",
    "/kaggle/working/ngram_full/work", "/kaggle/working/ngram_full/model",
    max_order=4, scan_chunk_tokens=1_000_000, run_occurrences_limit=2_000_000,
    merge_buffer_size=65_536, max_tokens=None, max_sentence_tokens=100)
result = t.run(resume=False, cleanup_runs=True)
print("DONE", result.report)
'''
open("/kaggle/working/run_train.py", "w").write(script)
import subprocess
log = open("/kaggle/working/train_log.txt", "w")
proc = subprocess.Popen(["python", "/kaggle/working/run_train.py"],
                        cwd="/kaggle/working/code", stdout=log, stderr=subprocess.STDOUT)
print("started, pid", proc.pid)

started, pid 146


In [ ]:
import os, time
p = "/kaggle/working/ngram_full/model/metadata.json"
while not os.path.isfile(p):
    time.sleep(15)
print("full model finished")

## 2. Evaluation and generation code (`lm_eval.py`)

In [19]:
!ls /kaggle/working/code /kaggle/working/code/src /kaggle/working/processed
!grep -n "^class \|    def \|^def " /kaggle/working/code/src/ngram_model.py

/kaggle/working/code:
app  data  notebooks  README.md  report  requirements.txt  results  src  tests

/kaggle/working/code/src:
ngram_disk.py	preprocess_lib.py  __pycache__
ngram_model.py	preprocess.py	   train_ngram_slice.py

/kaggle/working/processed:
dev.npy  stats.json  test.npy  train.npy  vocab.txt
29:def normalize_and_tokenize(text: str) -> list[str]:
34:def iter_encoded_sentences(flat_ids: Sequence[int] | np.ndarray) -> Iterable[list[int]]:
62:def _pack_windows(tokens: np.ndarray, order: int) -> np.ndarray:
71:def _pack_ids(token_ids: Sequence[int]) -> np.uint64:
79:class OrderCounts:
88:    def count(self, token_ids: Sequence[int]) -> int:
95:    def context_count(self, context: Sequence[int]) -> int:
103:    def next_counts(self, context: Sequence[int]) -> list[tuple[int, int]]:
118:class NGramModel:
121:    def __init__(self, vocabulary: Sequence[str], orders: dict[int, OrderCounts]):
133:    def fit(cls, sentences: Iterable[Sequence[int]], vocabulary: Sequence[str],
211:   

In [3]:
%%writefile /kaggle/working/lm_eval.py
"""Member 4: perplexity and random sentence generation for the repo's NGramModel.

Usage (in the Kaggle notebook, after the repo is cloned to /kaggle/working/code):
    import sys; sys.path.insert(0, "/kaggle/working"); sys.path.insert(0, "/kaggle/working/code")
    import lm_eval
"""
import math
import numpy as np

from src.ngram_model import BOS, EOS, UNK

K_GRID = [1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 0.1, 0.3, 1.0]
INTERP_WEIGHTS = [0.2, 0.4, 0.1, 0.3]  # tuned by the team on dev (full model)


# ----------------------------------------------------------------- data helpers
def sample_sentences(ids, n_sent, seed=42):
    """Random subset of complete <s> ... </s> sentences (all of them if n_sent is None)."""
    ids = np.asarray(ids)
    starts = np.flatnonzero(ids == BOS)
    ends = np.flatnonzero(ids == EOS)
    assert len(starts) == len(ends), "malformed sentence stream"
    if n_sent is None or n_sent >= len(starts):
        return ids
    rng = np.random.default_rng(seed)
    pick = np.sort(rng.choice(len(starts), n_sent, replace=False))
    return np.concatenate([ids[starts[i]:ends[i] + 1] for i in pick])


def remove_stopwords(ids, vocab_path, stopwords_path):
    """Same filtering that was used to build the no-stopword training data."""
    ids = np.asarray(ids)
    vocab = open(vocab_path, encoding="utf-8").read().splitlines()
    w2i = {w: i for i, w in enumerate(vocab)}
    stops = [w2i[w.strip()] for w in open(stopwords_path, encoding="utf-8").read().splitlines()
             if w.strip() in w2i and w2i[w.strip()] >= 4]
    is_stop = np.zeros(len(vocab), dtype=bool)
    is_stop[stops] = True
    f = ids[~is_stop[ids]]
    bos = np.flatnonzero(f == BOS)
    empty = bos[f[bos + 1] == EOS]            # sentences that became empty
    drop = np.zeros(len(f), dtype=bool)
    drop[empty] = True
    drop[empty + 1] = True
    return f[~drop]


# ------------------------------------------------------------ vectorised counts
def build_events(ids, order):
    """For every word/</s> in the stream: its n-1 context ids (BOS-padded) and its id."""
    ids = np.asarray(ids, dtype=np.int64)
    pos = np.arange(len(ids))
    is_bos = ids == BOS
    sent_start = np.maximum.accumulate(np.where(is_bos, pos, 0))
    target_idx = np.flatnonzero(~is_bos)
    ctx = []
    for d in range(order - 1, 0, -1):          # oldest context word first
        ctx.append(ids[np.maximum(target_idx - d, sent_start[target_idx])])
    return ctx, ids[target_idx]


def _pack(cols):
    key = np.zeros(len(cols[0]), dtype=np.uint64)
    for c in cols:
        key = (key << np.uint64(16)) | c.astype(np.uint64)
    return key


def _lookup(sorted_keys, values, query, chunk=1_000_000):
    out = np.zeros(len(query), dtype=np.float64)
    n = len(sorted_keys)
    for s in range(0, len(query), chunk):
        q = query[s:s + chunk]
        idx = np.searchsorted(sorted_keys, q)
        idx_c = np.minimum(idx, n - 1)
        hit = (idx < n) & (sorted_keys[idx_c] == q)
        part = np.zeros(len(q), dtype=np.float64)
        part[hit] = values[idx_c[hit]]
        out[s:s + chunk] = part
    return out


def event_counts(model, ids, order):
    """Return (count of n-gram, count of its context) for every prediction event."""
    o = model.orders[order]
    ctx, targets = build_events(ids, order)
    c = _lookup(o.keys, o.counts, _pack(ctx + [targets]))
    if order == 1:
        d = np.full(len(c), float(model.total_occurrences(1)))
    else:
        d = _lookup(o.context_keys, o.context_totals, _pack(ctx))
    return c, d


def ppl_addk(c, d, k, V):
    p = (c + k) / (d + k * V)
    return float(math.exp(-np.mean(np.log(p))))


def ppl_mle(c, d):
    zero = c == 0
    frac = float(zero.mean())
    ppl_seen = float(math.exp(-np.mean(np.log(c[~zero] / d[~zero]))))
    return (float("inf") if frac > 0 else ppl_seen), frac, ppl_seen


def tune_add_k(model, dev_ids, grid=K_GRID):
    """Best add-k per order on the DEVELOPMENT data (never the test set)."""
    V = len(model.allowed_outcomes)
    best = {}
    for n in sorted(model.orders):
        c, d = event_counts(model, dev_ids, n)
        scores = {k: ppl_addk(c, d, k, V) for k in grid}
        best[n] = min(scores, key=scores.get)
    return best


# --------------------------------------------------------- slow (repo prob()) ppl
def slow_ppl(model, ids, order, method, weights=None):
    """Perplexity through model.prob (backoff / interpolation). Slow: use a small sample."""
    ctx, targets = build_events(ids, order)
    total = 0.0
    for j in range(len(targets)):
        context = [int(col[j]) for col in ctx]
        p = model.prob(int(targets[j]), context, method=method, weights=weights)
        if p <= 0:
            return float("inf")
        total += math.log(p)
    return float(math.exp(-total / len(targets)))


# ------------------------------------------------------------------- evaluation
def evaluate_model(model, test_ids, dev_ids, label, n_test=50_000, n_dev=5_000,
                   n_slow=1_000, seed=42, do_slow=True):
    """Perplexity table for n=1..4. test_ids/dev_ids are flat <s>..</s> id streams."""
    import pandas as pd
    V = len(model.allowed_outcomes)
    dev = sample_sentences(dev_ids, n_dev, seed)
    test = sample_sentences(test_ids, n_test, seed)
    slow = sample_sentences(test_ids, n_slow, seed + 1)
    best_k = tune_add_k(model, dev)
    rows = []
    for n in sorted(model.orders):
        c, d = event_counts(model, test, n)
        mle, zero_frac, mle_seen = ppl_mle(c, d)
        row = {"model": label, "n": n, "events": len(c),
               "MLE": mle, "MLE_zero_prob_%": 100 * zero_frac,
               "best_k(dev)": best_k[n], "add_k": ppl_addk(c, d, best_k[n], V),
               "add_1": ppl_addk(c, d, 1.0, V)}
        if do_slow:
            row["backoff(sample)"] = slow_ppl(model, slow, n, "backoff")
            if n == 4:
                row["interp(sample)"] = slow_ppl(model, slow, 4, "interpolation", INTERP_WEIGHTS)
        rows.append(row)
    return pd.DataFrame(rows)


# ------------------------------------------------------------------- generation
def generate(model, order, n_sent=5, max_len=30, seed=0, allow_unk=False):
    """Random sentences from the order-n model, sampling from the observed counts.

    If the current n-1 word context was never seen, shorter contexts are used for that step.
    Returns (sentences, share_of_steps_that_needed_a_shorter_context).
    """
    rng = np.random.default_rng(seed)
    vocab = model.vocabulary
    o1 = model.orders[1]
    uni_ids = np.asarray(o1.keys, dtype=np.int64)
    uni_cnt = np.asarray(o1.counts, dtype=np.float64)
    if not allow_unk:
        uni_cnt = np.where(uni_ids == UNK, 0.0, uni_cnt)
    steps = shorter = 0

    def next_id(ctx):
        nonlocal shorter
        for m in range(order - 1, 0, -1):
            pairs = model.orders[m + 1].next_counts(ctx[len(ctx) - m:])
            if not allow_unk:
                pairs = [(t, c) for t, c in pairs if t != UNK]
            if pairs:
                if m < order - 1:
                    shorter += 1
                t = np.array([p[0] for p in pairs])
                w = np.array([p[1] for p in pairs], dtype=np.float64)
                return int(rng.choice(t, p=w / w.sum()))
        if order > 1:
            shorter += 1
        return int(rng.choice(uni_ids, p=uni_cnt / uni_cnt.sum()))

    out = []
    for _ in range(n_sent):
        ctx = [BOS] * (order - 1)
        words, ended = [], False
        for _ in range(max_len):
            t = next_id(ctx)
            steps += 1
            if t == EOS:
                ended = True
                break
            words.append(vocab[t])
            ctx = (ctx + [t])[-(order - 1):] if order > 1 else []
        out.append(" ".join(words) + ("" if ended else " ..."))
    return out, (shorter / steps if steps else 0.0)

Writing /kaggle/working/lm_eval.py


## 3. Perplexity (quick check on a test sample)

In [9]:
import os, sys, numpy as np
sys.path.insert(0, "/kaggle/working"); sys.path.insert(0, "/kaggle/working/code")
import lm_eval
from src.ngram_model import NGramModel
P = "/kaggle/working/processed"
dev, test = np.load(f"{P}/dev.npy"), np.load(f"{P}/test.npy")
print("test sentences:", int((test == 2).sum()))
full = NGramModel.load("/kaggle/working/ngram_full/model")
df_full = lm_eval.evaluate_model(full, test, dev, "full", n_test=20000, do_slow=False)
print(df_full.round(2).to_string())

test sentences: 504690
  model  n  events      MLE  MLE_zero_prob_%  best_k(dev)    add_k     add_1
0  full  1  317304  2401.56             0.00         0.00  2401.56   2401.60
1  full  2  317304      inf            13.53         0.01   535.14   1477.38
2  full  3  317304      inf            45.46         0.00  1892.52   9469.82
3  full  4  317304      inf            67.24         0.00  6447.12  21493.68


In [10]:
df_full = lm_eval.evaluate_model(full, test, dev, "full", n_test=20000, n_slow=300, do_slow=True)
pd_opts = {"display.width": 200, "display.max_columns": 20}
import pandas as pd
with pd.option_context(*[x for kv in pd_opts.items() for x in kv]):
    print(df_full.to_string(float_format=lambda v: f"{v:.4g}"))
df_full.to_csv("/kaggle/working/perplexity_full.csv", index=False)

  model  n  events  MLE  MLE_zero_prob_%  best_k(dev)  add_k     add_1  backoff(sample)  interp(sample)
0  full  1  317304 2402                0        1e-05   2402      2402             2392             NaN
1  full  2  317304  inf            13.53         0.01  535.1      1477            462.5             NaN
2  full  3  317304  inf            45.46        0.001   1893      9470            395.5             NaN
3  full  4  317304  inf            67.24       0.0003   6447 2.149e+04            384.5             309


## 4. Random sentences (5 per model)

In [11]:
for n in range(1, 5):
    sents, shorter = lm_eval.generate(full, n, n_sent=5, seed=7)
    print(f"\n=== n={n}  (steps needing a shorter context: {shorter:.0%})")
    print("\n".join(sents))


=== n=1  (steps needing a shorter context: 0%)
የዚህን ለስኬት ተለዋጭ በ<NUM> ድርጅት የፋሲካ
አሳታፊ ነጋዴው ላፊ ቤቱ ብለዋል ጥያቄ ባሻገር እንዲያውም ባለበት እወክላለሁ ሚኒስትርነት የፈጠራ ተሸናፊው እንዲሁም እንደ መላ

የሚችሉ ደስ ይጓዛል ፍጻሜ ቀድሞ ጉዳዩ ድረስ
ያለው ከጀመረ የኢትዮጵያ ሂደት

=== n=2  (steps needing a shorter context: 0%)
እሺ አትልም ብዬ መሬት እስከ ፒ <NUM> ኛው መላ ሀገሪቱን ለሁለት አመታት በፊት ሀምሌ <NUM> ስለዚህ ተኪ የለሽ ህጎች የአባትነት ሚና አለው
ይህ እና የሽግግር መንግስት ያወጣቸው መስፈርቶች ታማኝ ሚስት ሆነው ነው
በዚያም ቀን <NUM>
ኢትዮጵያዊት ነኝ
የባቢሎን ንጉስ ናቡከደነጾር በግዞት ከሚኖሩት ጋር ሲነጻጸር ገና ከአፉ የወጣ መልእክት ላይ የተመሰረተ በመሆኑ ጉዳይ መከታተል ጀመረ ቋንቋውን የማሳደግ ከባድ ጢአት የሰሩ ሰዎች የስልጣን ባለቤት ናት

=== n=3  (steps needing a shorter context: 4%)
እሺ አለችና ሄዳ ውጤት ኢትዮጵያን ለምንም ነገር ለሴት ለስልጣን አክብሮት ማሳየት ለሌሎች ሰዎች ጠቃሚ የሆኑ የሌሎች አገራትን ገድል ማውራት መስማትና ማንበብ እንደሆነ ይታመናል
ነገር ግን የአካባቢው አስተዳዳሪዎች የሀይማኖት መሪዎችና ፖለቲከኞች ውጭ ሆኖ ባለፈው ሳምንት ፖሊስ ማስታወቁ ተገልጿል
ይህን ያሉት በቃለ ምልልሱ ያነሳው
ከእነ አቶ መላኩ ፈንታ የገቢዎችና ጉምሩክ ባለስልጣን ጋር በመተባበር በአካባቢው ያለውን የቢዝነስ አማራጭ ከማየት ተቆጥበው ለችግሩ ብቸኛ መፍትሄ ግን የተፈጥሮ ሀብት ጥበቃ ስራ ጥገናና እድሳት የሚደረገው ደግሞ በተሽከርካሪ በመጓዝ ላይ ...
የነሱ ትልቁና ዋነኛው አላማቸው ትግራይ ከደርግ ሰራዊትና የመንግስ

## 5. Final perplexity table on the full test set and generated sentences (10 per model)

In [12]:
df_all = lm_eval.evaluate_model(full, test, dev, "full", n_test=None, do_slow=False)
df_all["backoff(300 sent.)"] = df_full["backoff(sample)"].values
df_all["interp(300 sent.)"] = df_full["interp(sample)"].values
print(df_all.to_string(float_format=lambda v: f"{v:.4g}"))
df_all.to_csv("/kaggle/working/perplexity_full_test.csv", index=False)

with open("/kaggle/working/generated_sentences.txt", "w", encoding="utf-8") as f:
    for n in range(1, 5):
        sents, shorter = lm_eval.generate(full, n, n_sent=10, seed=7)
        f.write(f"=== n={n} (shorter-context steps: {shorter:.0%})\n" + "\n".join(sents) + "\n\n")
print("saved")

  model  n   events  MLE  MLE_zero_prob_%  best_k(dev)  add_k    add_1  backoff(300 sent.)  interp(300 sent.)
0  full  1  8074537 2428                0        1e-05   2428     2428                2392                NaN
1  full  2  8074537  inf            13.58         0.01  538.3     1490               462.5                NaN
2  full  3  8074537  inf            45.66        0.001   1910     9517               395.5                NaN
3  full  4  8074537  inf            67.37       0.0003   6483 2.16e+04               384.5                309
saved


## 6. Save results

In [13]:
!cd /kaggle/working && zip -q member4_results.zip perplexity_full_test.csv perplexity_full.csv generated_sentences.txt lm_eval.py
!ls -lh /kaggle/working/member4_results.zip

-rw-r--r-- 1 root root 6.3K Oct 24 15:01 /kaggle/working/member4_results.zip


In [14]:
!unzip -l /kaggle/working/member4_results.zip

Archive:  /kaggle/working/member4_results.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
      509  2026-10-24 14:56   perplexity_full_test.csv
      501  2026-10-24 14:43   perplexity_full.csv
     5831  2026-10-24 14:56   generated_sentences.txt
     7893  2026-10-24 14:04   lm_eval.py
---------                     -------
    14734                     4 files


## 7. Are the generated sentences copied from the training data?

In [16]:
import numpy as np, re, glob
vocab = open("/kaggle/working/processed/vocab.txt", encoding="utf-8").read().splitlines()
w2i = {w: i for i, w in enumerate(vocab)}
tb = np.load("/kaggle/working/processed/train.npy").tobytes()

path = (glob.glob("/kaggle/working/generated_sentences.txt")
        or glob.glob("/kaggle/input/**/generated_sentences.txt", recursive=True))[0]
blocks = re.split(r"^=== n=(\d)[^\n]*\n", open(path, encoding="utf-8").read(), flags=re.M)[1:]
by_n = {int(blocks[i]): blocks[i + 1].split("\n") for i in range(0, len(blocks), 2)}

def occurs(sentence):
    ids = [w2i.get(w) for w in sentence.split()]
    if None in ids: return False
    pat = np.array([2] + ids + [3], dtype=np.int32).tobytes()
    start = 0
    while True:
        j = tb.find(pat, start)
        if j < 0: return False
        if j % 4 == 0: return True
        start = j + 1

for n in (2, 3, 4):
    sents = [s for s in by_n[n] if s.strip() and not s.endswith("...")]
    hits = sum(occurs(s) for s in sents)
    print(f"n={n}: {hits} of {len(sents)} complete sentences appear word for word in the training data")

n=2: 0 of 10 complete sentences appear word for word in the training data
n=3: 0 of 9 complete sentences appear word for word in the training data
n=4: 2 of 10 complete sentences appear word for word in the training data
